# 036_quality_check
Runs a check script. A check returns the rows that break its rule, so no rows means pass.
The result is always kept in a `dq_` table in Silver, so after a failure you can open it and
see exactly which accounts or figures are wrong.

A failed check fails the flow (without retrying) and every flow that depends on it is skipped:
Gold never builds on data that did not reconcile.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def quality_check(config, batch_id):
    df = run_script(config)
    failing_rows = write_table(df, "silver", config["destination_schema"], config["destination_table"])
    if failing_rows:
        table = table_full_name("silver", config["destination_schema"], config["destination_table"])
        raise QualityCheckFailed(f"{failing_rows} row(s) break {config['source_object']}; see {table}")
    return 0


LOADERS["036_quality_check"] = quality_check

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), quality_check, triggered_by="manual"))